# 인도 장애물 YOLO26n 학습 (Colab)

AI Hub [인도보행 영상](https://www.aihub.or.kr/aihubdata/data/view.do?dataSetSn=189)의 Bounding Box 데이터와 Hugging Face [ROD-Dataset](https://huggingface.co/datasets/Abtinzandi/Obstacle-Detection-Dataset-YOLO)으로 YOLO26n 을 파인튜닝하고 LiteRT(`.tflite`)로 내보낸다.

원본 라벨(AI Hub·ROD)은 안내 방식이 같은 것끼리 그룹(사람·이륜차·차·장애물·기둥·계단·보행 신호 빨간불)으로 묶어 학습한다. 박스는 그대로 두고 클래스 번호만 바꾸므로 다시 라벨링하지 않는다.

**처음 한 번 준비**
1. AI Hub 에서 인도보행 영상 데이터 사용 신청 → 승인
2. AI Hub > AI 개발지원 > 오픈 API 에서 API key 발급 (메일로 옴)
3. Colab 왼쪽 열쇠 아이콘(Secrets)에 `AIHUB_API_KEY` 로 넣고 노트북 접근 허용. **키를 코드에 쓰지 않는다.**
4. (선택) Hugging Face 토큰을 Secrets 에 `HF_TOKEN` 으로 넣는다. ROD 를 받을 때 요청 한도에 덜 걸린다.
5. 런타임 > 런타임 유형 변경 > **T4 GPU**

**흐름**: 다운로드(Colab 디스크) → YOLO 형식 변환·640px 축소 → 데이터셋을 Drive 에 보관 → (팀원 데이터 합치기 → 전문 모델로 빠진 라벨 채우기) → 학습(결과는 Drive) → `.tflite` 내보내기

**팀 학습 방식**: 각자 출처별(사람·장애물·신호등) 데이터로 전문 모델을 학습하고, 그 모델로 서로의 데이터에 빠진 라벨을 채운 뒤 모든 데이터를 합쳐 최종 모델 하나를 학습한다. 모델 가중치를 합치지 않는다.
- 셋 다 같은 `CLASSES` 순서로 학습한다.
- 전문 모델은 폰에 넣지 않으므로 `yolo26s.pt` 에서, 최종 모델은 `yolo26n.pt` 에서 새로 시작한다. 설정 셀이 `EXTRA_DATASET_ZIPS`·`TEACHERS` 가 있으면 최종, 없으면 전문으로 보고 자동으로 고른다.

세션이 끊기면 1번 설정 셀부터 다시 실행한다. Drive 에 데이터셋 zip 이 있으면 다운로드·변환을 건너뛰고, 학습은 `RESUME = True` 로 이어서 한다.

## 1. 설정

In [ ]:
from pathlib import Path

# AI Hub 인도보행 영상
DATASET_KEY = "189"
# 받을 Bbox 파일 key. `2. 파일 목록` 셀 결과에서 골라 넣는다. 하나(약 10GB)로 시작한다.
FILE_KEYS = ["49933"]  # Bbox_10_new.zip

# 데이터셋 버전. CLASSES·매핑·변환 코드를 바꾸면 올린다.
# 데이터셋 zip 과 학습 결과 폴더 이름에 붙어, 다른 버전의 데이터·결과와 섞이지 않는다.
DATASET_VERSION = "g7v1"

# 학습할 그룹. 순서가 곧 클래스 번호이므로 학습 뒤에는 바꾸지 않는다.
CLASSES = [
    "person",  # 사람: 멈춤·다가옴·빈 방향 계산
    "two_wheeler",  # 자전거·킥보드·오토바이: 빠르게 움직임
    "car",  # 차: 이면도로·인도 주차
    "obstacle",  # 바닥에 놓인 장애물: 볼라드·라바콘·벤치 등
    "pole",  # 가늘고 높은 것: 기둥·나무줄기
    "stairs",  # 계단: 넘어질 위험이 커서 따로 둔다
    "ped_signal_red",  # 보행 신호 빨간불: 신호등 데이터와 자동 라벨로 채운다
]

# AI Hub 라벨 이름(ROD 는 ROD_NAME_MAP 으로 바꾼 이름) → 그룹.
# 여기 없는 라벨의 박스는 버린다.
# 박스는 그대로 두고 번호만 바꾸므로 다시 라벨링하지 않는다.
LABEL_TO_CLASS = {
    # 사람
    "person": "person",
    # 이륜차
    "bicycle": "two_wheeler",
    "motorcycle": "two_wheeler",
    "scooter": "two_wheeler",
    # 차
    "car": "car",
    "bus": "car",
    "truck": "car",
    # 바닥에 놓인 장애물
    "bollard": "obstacle",
    "barricade": "obstacle",
    "movable_signage": "obstacle",
    "traffic_cone": "obstacle",
    "traffic_barrel": "obstacle",
    "guard_rail": "obstacle",
    "bench": "obstacle",
    "chair": "obstacle",
    "table": "obstacle",
    "potted_plant": "obstacle",
    "fire_hydrant": "obstacle",
    "power_controller": "obstacle",
    "traffic_light_controller": "obstacle",
    "kiosk": "obstacle",
    "dustbin": "obstacle",
    "bicycle_rack": "obstacle",
    # 사람이 밀거나 끄는 것. 미는 사람은 person 으로 따로 잡힌다
    "stroller": "obstacle",
    "wheelchair": "obstacle",
    "carrier": "obstacle",
    # 버스·택시 정류장
    "stop": "obstacle",
    # 가늘고 높은 것
    "pole": "pole",
    "tree_trunk": "pole",
    "parking_meter": "pole",
    # 계단
    "stairs": "stairs",
    # manhole: 바닥과 높이가 같아 장애물이 아니므로 버린다
}
assert set(LABEL_TO_CLASS.values()) <= set(CLASSES)

# ROD-Dataset (Hugging Face, MIT): 보행자 시점 장애물 25종, YOLO 형식
USE_ROD = True
ROD_REPO = "Abtinzandi/Obstacle-Detection-Dataset-YOLO"
# ROD 클래스 → AI Hub 라벨 이름. None 이거나 LABEL_TO_CLASS 에 없으면 버린다.
ROD_NAME_MAP = {
    "Bike": "bicycle",
    "Motorcycle": "motorcycle",
    "Person": "person",
    "Electrical Pole": "pole",
    # 나무 전체 박스라 AI Hub tree_trunk(줄기만)와 맞지 않는다
    "Tree": None,
    "Bench": "bench",
    "Chair": "chair",
    "Plant Pot": "potted_plant",
    "Fire hydrant": "fire_hydrant",
    "Electrical Box": "power_controller",
    "Stairs": "stairs",
    "Manhole": "manhole",
    "Traffic Cone": "traffic_cone",
    "Teraffic Barrel": "traffic_barrel",  # ROD 원본 철자 그대로
    "Dustbin": "dustbin",
    "Guard rail": "guard_rail",
    "Bicycle Rack": "bicycle_rack",
    # 차는 LABEL_TO_CLASS 에서 car 그룹으로 묶는다
    "Car": "car",
    "Truck": "truck",
    "Bus": "bus",
    # 장애물이 아니거나 높은 곳에 있는 것
    "Building": None,
    "Road": None,
    "Pedestrian crosswalk": None,
    "Traffic sign": None,
    "Dog": None,
}
# 이 클래스가 있는 ROD 사진은 통째로 뺀다.
# Tree 박스만 버리면 사진 속 나무줄기가 배경으로 학습되어
# tree_trunk → pole 과 반대로 학습된다.
ROD_DROP_IMAGE_CLASSES = ["Tree"]
# HF 요청 한도(5분 단위)에 걸렸을 때 다시 받는 횟수와 기다리는 시간(초)
ROD_MAX_RETRIES = 5
ROD_RETRY_WAIT_S = 300

IMG_SIZE = 640  # 학습 입력 크기. 이미지도 긴 변을 이 크기로 줄여 저장한다
VAL_RATIO = 0.1  # 검증셋 비율 (XML 묶음 단위)
EPOCHS = 100
BATCH = 16
RESUME = False  # 세션이 끊겨 이어서 학습할 때 True

RAW_DIR = Path("/content/raw")  # 원본(세션 끝나면 사라짐)
ROD_DIR = Path("/content/rod")
DATA_DIR = Path("/content/sidewalk_yolo")  # 변환 결과
DRIVE_DIR = Path("/content/drive/MyDrive/saitgil")

# 팀원이 만든 데이터셋 zip(Drive 경로). 같은 CLASSES 로 만든 것만 합친다.
EXTRA_DATASET_ZIPS = [
    # DRIVE_DIR / "person_yolo_g7v1.zip",
]
# 빠진 라벨을 채울 전문 모델: 같은 CLASSES 순서로 학습한 best.pt 와 채울 클래스.
TEACHERS = [
    # {
    #     "weights": DRIVE_DIR / "runs/person_g7v1_yolo26s_v1/weights/best.pt",
    #     "classes": ["person"],
    # },
]
AUTOLABEL_CONF = 0.5  # 이 확신도 이상인 박스만 채운다
AUTOLABEL_IOU = 0.5  # 같은 클래스의 기존 박스와 이만큼 겹치면 이미 있는 것으로 본다
AUTOLABEL_SAMPLES = 30  # 검수용으로 그려 저장할 사진 수

# 학습 시작 모델은 단계에 따라 자동으로 고른다. 손으로 바꾸지 않는다.
# 팀원 데이터를 합치거나 전문 모델로 라벨을 채우면 폰에 넣을 최종 모델,
# 아니면 자동 라벨에 쓸 전문 모델(폰에 넣지 않으므로 더 큰 모델)을 학습한다.
SPECIALIST_WEIGHTS = "yolo26s.pt"
FINAL_WEIGHTS = "yolo26n.pt"
IS_FINAL = bool(EXTRA_DATASET_ZIPS or TEACHERS)
BASE_WEIGHTS = FINAL_WEIGHTS if IS_FINAL else SPECIALIST_WEIGHTS
# 이 노트북이 맡은 전문 분야. 팀원끼리 같은 Drive 에 저장해도
# 결과·데이터셋이 겹치지 않게 이름 앞에 붙는다. 최종 모델은 final 을 붙인다.
SPECIALTY = "obstacle"
RUN_PREFIX = "final" if IS_FINAL else SPECIALTY
# 모델 크기도 붙여 전문 모델과 최종 모델 결과가 섞이지 않게 한다.
RUN_NAME = f"{RUN_PREFIX}_{DATASET_VERSION}_{Path(BASE_WEIGHTS).stem}_v1"
print("학습 단계:", "최종" if IS_FINAL else "전문", "| 시작 모델:", BASE_WEIGHTS)
print("결과 폴더:", RUN_NAME)

ZIP_NAME = "_".join(FILE_KEYS) + ("_rod" if USE_ROD else "") + f"_{DATASET_VERSION}"
DATASET_ZIP = DRIVE_DIR / f"{SPECIALTY}_yolo_{ZIP_NAME}.zip"
RUNS_DIR = DRIVE_DIR / "runs"
AUTOLABEL_SAMPLE_DIR = DRIVE_DIR / "autolabel_samples" / RUN_NAME

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics

from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## 2. AI Hub 에서 받기

Drive 에 변환된 데이터셋이 있으면 이 단계와 3단계를 건너뛰고 4단계로 간다.

In [ ]:
import os

from google.colab import userdata

# 키는 Secrets 에서 읽어 환경변수로만 넘긴다(출력하지 않음).
os.environ["AIHUB_API_KEY"] = userdata.get("AIHUB_API_KEY")

!curl -s -o /usr/local/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do
!chmod +x /usr/local/bin/aihubshell

In [ ]:
# 파일 목록: Bbox_xx_new.zip 줄 끝 숫자가 filekey 다.
!aihubshell -mode l -datasetkey {DATASET_KEY} | grep -i -E "bbox|라벨|xml"

In [ ]:
DOWNLOAD_ARGS = f"-mode d -datasetkey {DATASET_KEY} -filekey {','.join(FILE_KEYS)}"
# 다운로드가 끝나면 aihubshell 이 압축을 풀고 zip 을 지운다.
# 받는 용량의 2~3배 디스크가 필요하다.
if DATASET_ZIP.exists():
    print("Drive 에 데이터셋이 있어 받지 않습니다:", DATASET_ZIP)
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    %cd {RAW_DIR}
    !aihubshell {DOWNLOAD_ARGS} -aihubapikey "$$AIHUB_API_KEY"
    %cd /content
    !df -h /content | tail -1
    !find {RAW_DIR} -name "*.xml" | head -3
    !find {RAW_DIR} -name "*.xml" | wc -l

## 3. YOLO 형식으로 바꾸기

AI Hub CVAT XML 의 `box`(px 좌표)를 YOLO `class cx cy w h`(0~1)로 바꾸고, 이미지는 긴 변 640px JPEG 로 줄인다. ROD 는 이미 YOLO 형식이라 클래스 번호만 바꾼다. 라벨 이름은 `LABEL_TO_CLASS` 로 그룹 번호가 되고, 매핑에 없는 라벨은 버린다.

In [ ]:
import random
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def index_images(root: Path) -> dict[str, list[Path]]:
    """파일 이름 → 경로들.

    XML 의 image name 은 폴더가 빠져 있을 수 있어 이름으로 찾는다.
    폴더가 달라도 이름이 같을 수 있어 경로를 모두 남긴다.
    """
    images = defaultdict(list)
    for p in root.rglob("*"):
        if p.suffix.lower() in IMAGE_SUFFIXES:
            images[p.name].append(p)
    return images


def find_image(name: str, xml_path: Path, images: dict[str, list[Path]]) -> list[Path]:
    """XML 에 가까운 폴더부터 올라가며 이름이 같은 사진을 찾는다.

    하나로 좁혀지면 그 하나를, 못 좁히면 후보를 모두 돌려준다(빈 목록이면 없음).
    """
    candidates = images.get(name, [])
    if len(candidates) <= 1:
        return candidates
    for folder in [xml_path.parent, *xml_path.parent.parents]:
        near = [p for p in candidates if folder in p.parents]
        if near:
            return near
    return candidates


def yolo_lines(image: ET.Element, class_ids: dict[str, int]) -> list[str]:
    """CVAT box(xtl, ytl, xbr, ybr, px) → YOLO 'class cx cy w h' (0~1)."""
    width, height = float(image.get("width")), float(image.get("height"))
    lines = []
    for box in image.iter("box"):
        label = box.get("label")
        if label not in class_ids:
            continue
        xtl, ytl = max(float(box.get("xtl")), 0), max(float(box.get("ytl")), 0)
        xbr, ybr = min(float(box.get("xbr")), width), min(float(box.get("ybr")), height)
        if xbr <= xtl or ybr <= ytl:
            continue
        cx, cy = (xtl + xbr) / 2 / width, (ytl + ybr) / 2 / height
        w, h = (xbr - xtl) / width, (ybr - ytl) / height
        lines.append(f"{class_ids[label]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines


def polygon_to_box(coords: list[str]) -> str | None:
    """YOLO 폴리곤 'x1 y1 x2 y2 ...'(0~1) → 감싸는 박스 'cx cy w h'."""
    xs = [min(max(float(v), 0), 1) for v in coords[0::2]]
    ys = [min(max(float(v), 0), 1) for v in coords[1::2]]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    if x1 <= x0 or y1 <= y0:
        return None
    return f"{(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {x1 - x0:.6f} {y1 - y0:.6f}"


def save_resized(src: Path, dst: Path, long_side: int) -> None:
    """긴 변을 long_side 로 줄여 JPEG 로 저장한다. 라벨은 0~1 비율이라 그대로 맞는다."""
    with Image.open(src) as img:
        img = img.convert("RGB")
        img.thumbnail((long_side, long_side))
        img.save(dst, "JPEG", quality=90)


def convert(
    raw_dir: Path,
    out_dir: Path,
    classes: list[str],
    label_to_class: dict[str, str],
    long_side: int,
    val_ratio: float,
    seed: int = 0,
) -> Counter:
    """AI Hub 인도보행 Bbox(CVAT XML) → YOLO 데이터셋.

    같은 XML(같은 촬영 묶음)의 사진은 거의 같아,
    검증셋은 사진이 아니라 XML 단위로 나눈다.
    라벨 이름은 label_to_class 로 그룹 번호로 바꾸고, 없는 라벨은 버린다.
    사진을 하나로 못 좁히거나 결과 이름이 겹치면 건너뛰고 개수를 센다.
    """
    class_ids = {label: classes.index(group) for label, group in label_to_class.items()}
    images = index_images(raw_dir)
    xmls = sorted(raw_dir.rglob("*.xml"))
    random.Random(seed).shuffle(xmls)
    n_val = max(1, round(len(xmls) * val_ratio)) if len(xmls) > 1 else 0
    counts: Counter = Counter()
    counts["same_name_files"] = sum(len(p) for p in images.values() if len(p) > 1)
    for i, xml_path in enumerate(xmls):
        split = "val" if i < n_val else "train"
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        # 폴더마다 XML 이름이 같을 수 있어 raw_dir 기준 경로를 이름에 쓴다.
        xml_key = "_".join(xml_path.relative_to(raw_dir).with_suffix("").parts)
        for image in ET.parse(xml_path).getroot().iter("image"):
            found = find_image(Path(image.get("name")).name, xml_path, images)
            if not found:
                counts["missing_image"] += 1
                continue
            if len(found) > 1:
                counts["ambiguous_image"] += 1
                continue
            src = found[0]
            stem = f"{xml_key}_{src.stem}"
            dst = out_dir / "images" / split / f"{stem}.jpg"
            if dst.exists():
                counts["duplicate_output"] += 1
                continue
            lines = yolo_lines(image, class_ids)
            save_resized(src, dst, long_side)
            (out_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
            counts[f"{split}_images"] += 1
            counts.update(classes[int(line.split()[0])] for line in lines)
    (out_dir / "data.yaml").write_text(
        f"path: {out_dir}\ntrain: images/train\nval: images/val\n"
        + "names:\n"
        + "".join(f"  {i}: {name}\n" for i, name in enumerate(classes))
    )
    return counts


def add_yolo_dataset(
    src_dir: Path,
    out_dir: Path,
    src_names: list[str],
    name_map: dict[str, str | None],
    classes: list[str],
    label_to_class: dict[str, str],
    long_side: int,
    prefix: str,
    drop_image_classes: list[str],
) -> Counter:
    """이미 YOLO 형식인 데이터셋(train/valid)을 클래스 번호를 바꿔 out_dir 에 더한다.

    name_map 으로 AI Hub 라벨 이름으로 바꾼 뒤 label_to_class 로 그룹 번호를 정한다.
    name_map 에 없거나 None 이거나 label_to_class 에 없는 클래스의 박스는 버린다.
    images/ 아래 하위 폴더까지 찾고, 라벨은 labels/ 아래 같은 경로에서 찾는다.
    폴리곤 줄은 감싸는 박스로 바꾼다.
    drop_image_classes 가 하나라도 있는 사진은 통째로 뺀다.
    """
    class_ids = {label: classes.index(group) for label, group in label_to_class.items()}
    remap = {
        i: class_ids[name_map[name]]
        for i, name in enumerate(src_names)
        if name_map.get(name) in class_ids
    }
    drop_ids = {i for i, name in enumerate(src_names) if name in drop_image_classes}
    counts: Counter = Counter()
    for src_split, split in [("train", "train"), ("valid", "val")]:
        images_dir = src_dir / src_split / "images"
        labels_dir = src_dir / src_split / "labels"
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        for src in sorted(images_dir.rglob("*")):
            if src.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            rel = src.relative_to(images_dir).with_suffix("")
            label = (labels_dir / rel).with_suffix(".txt")
            rows = []
            if label.exists():
                rows = [line.split() for line in label.read_text().splitlines()]
                rows = [parts for parts in rows if parts]
            if any(int(parts[0]) in drop_ids for parts in rows):
                counts["dropped_images"] += 1
                continue
            lines = []
            for parts in rows:
                class_id = int(parts[0])
                if class_id not in remap:
                    continue
                if len(parts) == 5:
                    box = " ".join(parts[1:])
                elif len(parts) >= 7 and len(parts) % 2 == 1:
                    box = polygon_to_box(parts[1:])
                    counts["polygon_boxes"] += 1
                else:
                    box = None
                if box is not None:
                    lines.append(f"{remap[class_id]} {box}")
            stem = "_".join([prefix, *rel.parts])
            save_resized(src, out_dir / "images" / split / f"{stem}.jpg", long_side)
            (out_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
            counts[f"{split}_images"] += 1
            counts.update(classes[int(line.split()[0])] for line in lines)
    return counts

In [ ]:
if DATASET_ZIP.exists():
    print("Drive 에 데이터셋이 있어 변환하지 않습니다.")
else:
    counts = convert(RAW_DIR, DATA_DIR, CLASSES, LABEL_TO_CLASS, IMG_SIZE, VAL_RATIO)
    for key in [
        "train_images",
        "val_images",
        "missing_image",
        "ambiguous_image",
        "duplicate_output",
        "same_name_files",
    ]:
        print(key, counts.pop(key, 0))
    print("AI Hub 그룹별 박스 수:", dict(counts.most_common()))
    # 원본은 지워 디스크를 비운다.
    !rm -rf {RAW_DIR}

### ROD 더하기

ROD-Dataset(약 1.5GB)을 받아 AI Hub 결과에 더한다. 파일 이름 앞에 `rod_` 를 붙인다.

- `train/images/0`, `train/images/1` 처럼 하위 폴더까지 찾는다.
- 폴리곤 라벨은 감싸는 박스로 바꾼다.
- `ROD_DROP_IMAGE_CLASSES`(Tree)가 있는 사진은 뺀다.
- Colab Secrets 에 `HF_TOKEN` 이 있으면 쓴다. 요청 한도에 걸리면 기다렸다가 이어받는다.

ROD 는 공개 데이터 26개를 합친 것이라 사진의 96% 가 클래스 하나만 라벨링돼 있다. 사진 속 다른 물체(사람·기둥 등)는 배경으로 학습되므로, 최종 학습 전에 팀 모델로 자동 라벨을 붙여 채운다.

In [ ]:
import time

import yaml
from google.colab import userdata
from huggingface_hub import snapshot_download

if DATASET_ZIP.exists() or not USE_ROD:
    print("ROD 를 더하지 않습니다.")
else:
    # HF_TOKEN 이 Secrets 에 없거나 접근을 허용하지 않았으면 토큰 없이 받는다.
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
    print("HF_TOKEN 사용:", hf_token is not None)
    # 요청 한도(429)에 걸리면 기다렸다가 같은 폴더로 이어받는다.
    for attempt in range(1, ROD_MAX_RETRIES + 1):
        try:
            snapshot_download(
                ROD_REPO, repo_type="dataset", local_dir=ROD_DIR, token=hf_token
            )
            break
        except Exception as error:
            print(f"ROD 받기 실패 {attempt}/{ROD_MAX_RETRIES}: {error}")
            if attempt == ROD_MAX_RETRIES:
                raise
            time.sleep(ROD_RETRY_WAIT_S)
    rod_names = yaml.safe_load((ROD_DIR / "data.yaml").read_text())["names"]
    counts = add_yolo_dataset(
        ROD_DIR,
        DATA_DIR,
        rod_names,
        ROD_NAME_MAP,
        CLASSES,
        LABEL_TO_CLASS,
        IMG_SIZE,
        "rod",
        ROD_DROP_IMAGE_CLASSES,
    )
    for key in ["train_images", "val_images", "dropped_images", "polygon_boxes"]:
        print(key, counts.pop(key, 0))
    print("ROD 그룹별 박스 수:", dict(counts.most_common()))
    !rm -rf {ROD_DIR}

### Drive 에 보관

In [ ]:
if DATASET_ZIP.exists():
    print("이미 있습니다:", DATASET_ZIP)
else:
    # 줄인 데이터셋을 Drive 에 보관한다.
    !cd /content && zip -q -r {DATASET_ZIP} {DATA_DIR.name}
    !ls -lh {DATASET_ZIP}

## 4. 데이터셋 준비 (Drive 에서 복원·팀원 데이터 합치기)

내 데이터셋 zip 을 풀고, `EXTRA_DATASET_ZIPS` 의 팀원 데이터셋을 더한다. 클래스 순서가 `CLASSES` 와 다른 zip 이면 멈춘다.

In [ ]:
import shutil
import tempfile
from collections import Counter

import yaml


def read_names(yaml_path: Path) -> list[str]:
    """data.yaml 의 names 를 번호 순서 목록으로 읽는다."""
    names = yaml.safe_load(yaml_path.read_text())["names"]
    if isinstance(names, dict):
        return [names[i] for i in sorted(names)]
    return list(names)


def merge_dataset(zip_path: Path, out_dir: Path, classes: list[str]) -> Counter:
    """팀원 데이터셋 zip 의 train·val 을 out_dir 에 더한다.

    zip 안 data.yaml 의 클래스가 classes 와 다르면 멈춘다.
    이름이 같은 사진은 이미 있는 것으로 보고 건너뛴다.
    """
    counts: Counter = Counter()
    with tempfile.TemporaryDirectory() as tmp:
        shutil.unpack_archive(zip_path, tmp)
        src_yaml = next(Path(tmp).rglob("data.yaml"))
        if read_names(src_yaml) != classes:
            raise RuntimeError(
                f"{zip_path.name} 의 클래스 {read_names(src_yaml)} 가 "
                f"CLASSES {classes} 와 다릅니다. 같은 CLASSES 로 다시 만듭니다."
            )
        src_dir = src_yaml.parent
        for split in ["train", "val"]:
            (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
            (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
            for src in sorted((src_dir / "images" / split).glob("*")):
                dst = out_dir / "images" / split / src.name
                if dst.exists():
                    counts["duplicate_skipped"] += 1
                    continue
                shutil.copy2(src, dst)
                label = (src_dir / "labels" / split / src.name).with_suffix(".txt")
                out_label = (out_dir / "labels" / split / src.name).with_suffix(".txt")
                if label.exists():
                    shutil.copy2(label, out_label)
                counts[f"{split}_images"] += 1
    return counts


if not (DATA_DIR / "data.yaml").exists():
    !cd /content && unzip -q {DATASET_ZIP}
# data.yaml 의 path 를 지금 위치로 맞춘다.
yaml_path = DATA_DIR / "data.yaml"
lines = yaml_path.read_text().splitlines()
yaml_path.write_text("\n".join([f"path: {DATA_DIR}"] + lines[1:]) + "\n")
print(yaml_path.read_text())

# 다른 클래스 체계로 만든 데이터셋이면 멈춘다.
names = read_names(yaml_path)
if names != CLASSES:
    raise RuntimeError(
        f"data.yaml 클래스 {names} 가 CLASSES {CLASSES} 와 다릅니다. "
        "DATASET_VERSION 을 올리거나 Drive 의 데이터셋 zip 을 지우고 다시 변환합니다."
    )

# 팀원 데이터셋을 더한다. 다시 실행해도 이미 있는 사진은 건너뛴다.
for zip_path in EXTRA_DATASET_ZIPS:
    counts = merge_dataset(Path(zip_path), DATA_DIR, CLASSES)
    print(zip_path, dict(counts))
for split in ["train", "val"]:
    print(split, "사진 수:", len(list((DATA_DIR / "images" / split).glob("*"))))

## 5. 빠진 라벨 채우기 (자동 라벨)

`TEACHERS` 의 전문 모델로 **학습셋(train)** 사진에 빠진 클래스 박스를 채운다. 검증셋(val)은 손으로 단 라벨만 두어 점수를 믿을 수 있게 한다.

- 전문 모델은 같은 `CLASSES` 순서로 학습한 것이어야 한다. 번호가 다르면 멈춘다.
- 확신도 `AUTOLABEL_CONF` 이상만 넣고, 같은 클래스의 기존 박스와 `AUTOLABEL_IOU` 이상 겹치면 넣지 않는다.
- 채운 라벨은 Colab 디스크에만 있으므로 세션마다 4번 복원 뒤 다시 실행한다. `TEACHERS` 가 비어 있으면 건너뛴다.
- **학습 전에** Drive `autolabel_samples/` 에 저장된 사진을 열어 박스가 맞는지 확인한다. 틀린 박스가 많으면 `AUTOLABEL_CONF` 를 올린다.

In [ ]:
from collections import Counter

from PIL import Image  # noqa: F811 (3번 셀을 건너뛰어도 이 셀만으로 돈다)
from ultralytics import YOLO


def to_corners(box: list[float]) -> tuple[float, float, float, float]:
    """YOLO 'cx cy w h' → 'x0 y0 x1 y1'."""
    cx, cy, w, h = box
    return cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2


def box_iou(a: list[float], b: list[float]) -> float:
    """YOLO 'cx cy w h'(0~1) 두 박스가 겹치는 정도(0~1)."""
    ax0, ay0, ax1, ay1 = to_corners(a)
    bx0, by0, bx1, by1 = to_corners(b)
    inter_w = max(0.0, min(ax1, bx1) - max(ax0, bx0))
    inter_h = max(0.0, min(ay1, by1) - max(ay0, by0))
    inter = inter_w * inter_h
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0


def autolabel(
    data_dir: Path,
    teachers: list[dict],
    classes: list[str],
    conf: float,
    iou: float,
    imgsz: int,
    sample_dir: Path,
    n_samples: int,
) -> Counter:
    """전문 모델로 학습셋(train) 라벨에 빠진 박스를 더한다. 검증셋은 건드리지 않는다.

    teacher["classes"] 의 클래스만 채우고, 같은 클래스의 기존 박스와
    iou 이상 겹치면 이미 있는 것으로 보고 넣지 않는다(다시 실행해도 겹쳐 쌓이지 않는다).
    박스를 더한 사진 중 n_samples 장은 검수용으로 sample_dir 에 그려 저장한다.
    """
    images_dir = data_dir / "images" / "train"
    labels_dir = data_dir / "labels" / "train"
    sample_dir.mkdir(parents=True, exist_ok=True)
    counts: Counter = Counter()
    for teacher in teachers:
        model = YOLO(str(teacher["weights"]))
        names = [model.names[i] for i in sorted(model.names)]
        if names != classes:
            raise RuntimeError(
                f"{teacher['weights']} 의 클래스 {names} 가 CLASSES 와 다릅니다. "
                "같은 CLASSES 순서로 학습한 모델만 쓴다."
            )
        target_ids = sorted(classes.index(name) for name in teacher["classes"])
        results = model.predict(
            source=str(images_dir),
            stream=True,
            conf=conf,
            imgsz=imgsz,
            classes=target_ids,
            verbose=False,
        )
        for result in results:
            label = labels_dir / f"{Path(result.path).stem}.txt"
            text = label.read_text() if label.exists() else ""
            rows = [line.split() for line in text.splitlines() if line.strip()]
            boxes = [(int(r[0]), [float(v) for v in r[1:5]]) for r in rows]
            added = []
            predicted = zip(result.boxes.cls.tolist(), result.boxes.xywhn.tolist())
            for class_id, box in predicted:
                class_id = int(class_id)
                if any(c == class_id and box_iou(b, box) >= iou for c, b in boxes):
                    counts["overlap_skipped"] += 1
                    continue
                boxes.append((class_id, box))
                added.append((class_id, box))
            if not added:
                continue
            new_lines = [
                f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, (x, y, w, h) in added
            ]
            label.write_text("\n".join([" ".join(r) for r in rows] + new_lines))
            counts["images_changed"] += 1
            counts.update(classes[c] for c, _ in added)
            if counts["samples"] < n_samples:
                # plot() 은 BGR 배열이라 RGB 로 뒤집어 저장한다.
                Image.fromarray(result.plot()[..., ::-1]).save(
                    sample_dir / label.with_suffix(".jpg").name
                )
                counts["samples"] += 1
    return counts


if not TEACHERS:
    print("TEACHERS 가 비어 있어 자동 라벨을 건너뜁니다.")
else:
    counts = autolabel(
        DATA_DIR,
        TEACHERS,
        CLASSES,
        AUTOLABEL_CONF,
        AUTOLABEL_IOU,
        IMG_SIZE,
        AUTOLABEL_SAMPLE_DIR,
        AUTOLABEL_SAMPLES,
    )
    for key in ["images_changed", "overlap_skipped", "samples"]:
        print(key, counts.pop(key, 0))
    print("채운 클래스별 박스 수:", dict(counts.most_common()))
    print("검수용 사진:", AUTOLABEL_SAMPLE_DIR)

## 6. 학습

결과(`best.pt`, `last.pt`, 그래프)는 Drive `saitgil/runs/<RUN_NAME>/` 에 쌓인다. 같은 이름의 폴더가 이미 있으면 덮어쓰지 않고 멈춘다. 세션이 끊기면 `RESUME = True` 로 바꾸고 1·4·5·6 셀을 다시 실행한다.

In [ ]:
from ultralytics import YOLO  # noqa: F811 (이어서 학습할 때 이 셀만으로 돈다)

last = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
if RESUME and last.exists():
    model = YOLO(str(last))
    model.train(resume=True)
else:
    # 같은 이름의 결과가 있으면 덮어쓰지 않고 멈춘다(팀원 결과일 수 있다).
    if (RUNS_DIR / RUN_NAME).exists():
        raise RuntimeError(
            f"{RUNS_DIR / RUN_NAME} 가 이미 있습니다. 이어서 학습하려면 RESUME = True, "
            "새로 학습하려면 설정 셀 RUN_NAME 끝의 _v1 을 _v2 로 올립니다."
        )
    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(DATA_DIR / "data.yaml"),
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        patience=20,
    )

## 7. 검증

In [ ]:
best = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
metrics = YOLO(str(best)).val(data=str(DATA_DIR / "data.yaml"), imgsz=IMG_SIZE)
print("mAP50:", round(metrics.box.map50, 3), "mAP50-95:", round(metrics.box.map, 3))
# ap50 은 검증셋에 나온 클래스만 ap_class_index 순서로 담긴다.
for class_id, ap50 in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"{CLASSES[class_id]:26s} AP50 {ap50:.3f}")

## 8. LiteRT(`.tflite`)로 내보내기

받은 `.tflite` 는 커밋하지 않고 앱 assets(`frontend/android/app/src/main/assets/models/`)에 복사한다.

In [ ]:
exported = YOLO(str(best)).export(format="tflite", imgsz=IMG_SIZE)
print(exported)
!cp -r {Path(exported).parent} {RUNS_DIR / RUN_NAME}/
!ls -lh {RUNS_DIR / RUN_NAME}